# Week 3: RFI-family weakness analysis
Author: Lei; repo adaptation retains native-count metrics and recording-level bootstrap.

Which RFI families cause missed interference and false flags for the best CNN? Changes from Week 2: break the same frozen validation evaluation down by family. No retraining or threshold changes. This copy reports rates from saved counts; per-family score histograms are unavailable in the reference archive, so per-family AP is not recomputed.


In [ ]:
from cs163_rfi import paths
from cs163_rfi.family import *
RUN_DIR = paths.RESULTS_DIR / "reference/family_source"
EVALUATION_DIR = RUN_DIR
OUTPUT_DIR = paths.RESULTS_DIR / "week_03/family_analysis"


In [ ]:
run = RUN_DIR
out = OUTPUT_DIR
metadata = json.loads((EVALUATION_DIR / 'run_metadata.json').read_text())
summary = json.loads((EVALUATION_DIR / 'summary.json').read_text())
calibration = summary['calibration']
source = read_csv(EVALUATION_DIR / 'per_antenna.csv')
families = sorted({r['rfi_type'] for r in source} - {'clean', 'sky'})
all_families = families + ['clean', 'sky']
methods = sorted({r['method'] for r in source})
rng = np.random.default_rng(163)
result = []
names = ['precision', 'recall', 'f1', 'iou', 'fpr', 'miss_rate']
for family in all_families:
    for method in methods:
        selected = [r for r in source if r['rfi_type'] == family and r['method'] == method]
        groups = {}
        for r in selected:
            groups.setdefault(r['sample_id'], np.zeros(4, dtype=np.int64))
            groups[r['sample_id']] += np.array([int(r[k]) for k in COUNTS])
        if not groups:
            raise ValueError(f'Missing {family}/{method}')
        counts = np.stack(list(groups.values()))
        total = counts.sum(axis=0)
        values = metrics(total)
        row = dict(rfi_type=family, method=method, recordings=len(groups), antennas=len(selected),
                   **dict(zip(COUNTS, total.tolist())), **dict(zip(names, values.tolist())))
        row['prevalence'] = float(divide(total[0]+total[2], total.sum()))
        if method == MODEL:
            draws = rng.integers(0, len(counts), (2000, len(counts)))
            boot = metrics(counts[draws].sum(axis=1))
            for i, name in enumerate(names):
                finite = boot[:, i][np.isfinite(boot[:, i])]
                lo, hi = np.percentile(finite, [2.5, 97.5]) if len(finite) else (np.nan, np.nan)
                row[name+'_ci_low'], row[name+'_ci_high'] = float(lo), float(hi)
        result.append(row)
out.mkdir(parents=True, exist_ok=True)
# Give all CSV rows the same schema, including baseline rows without CIs.
fields = list(dict.fromkeys(k for r in result for k in r))
write_csv(out/'family_metrics.csv', [{k:r.get(k, '') for k in fields} for r in result])
model = {r['rfi_type']:r for r in result if r['method'] == MODEL}
plt.rcParams.update({'font.size':10, 'axes.spines.top':False, 'axes.spines.right':False})
def save(fig, name):
    fig.savefig(out/(name+'.png'), dpi=180, bbox_inches='tight')
    display(Image(filename=str(out/(name+'.png'))))
    plt.close(fig)


## Saved decision threshold
A positive prediction means **RFI**. The saved rule is `floor(sigmoid(logit) × 65535) >= 1786`, mathematically equivalent to a sigmoid score **at least 1786 / 65535 ≈ 0.0272526**. Evaluation preserves the quantized rule.

The cutoff was calibrated on three clean validation recordings to match the approximately 1.224% clean false-positive rate of `sk-bandpass`. The sigmoid scores are not necessarily calibrated probabilities; the cutoff is not a confidence guarantee.

In [ ]:
print(f"Split: {metadata['evaluation_split']} | Selected epoch: {metadata['selected_epoch']}")
print(f"Overall pixel AP: {summary['pixel_ap_quantized']:.4f}")
print(f"Saved score threshold: {calibration['probability_lower_bound']:.7f}")

## 1. Validation dataset pixel breakdown

This figure matches the requested three-panel layout, restricted to the validation data actually analyzed:

1. **Pixel locations per family**, including both RFI and non-RFI.
2. **RFI prevalence within each family:** (TP+FN)/(TP+FP+FN+TN).
3. **Each family's share of all validation pixel locations.**

One method contributes each sample/antenna once; counts are not multiplied by the number of methods. Each recording contains 4096 × 10 × 128 = 5,242,880 pixel locations. Families label recordings, so mixed does not isolate individual emitter types. Counts and prevalence differ from an all-splits dataset summary.

In [ ]:
# One method supplies each sample/antenna exactly once; do not sum methods.
breakdown = []
total_pixels = sum(sum(model[f][k] for k in COUNTS) for f in all_families)
for f in sorted(all_families, key=lambda f: (-sum(model[f][k] for k in COUNTS), f)):
    r = model[f]
    pixels = sum(r[k] for k in COUNTS)
    breakdown.append(dict(rfi_type=f, recordings=r['recordings'],
                          pixel_locations=pixels, rfi_pixels=r['tp']+r['fn'],
                          rfi_fraction=(r['tp']+r['fn'])/pixels,
                          dataset_fraction=pixels/total_pixels))
write_csv(out/'dataset_breakdown.csv', breakdown)
fig, axes = plt.subplots(1, 3, figsize=(17, 7), sharey=True)
for ax, field, factor, title, xlabel, color in zip(
        axes, ['pixel_locations','rfi_fraction','dataset_fraction'], [1e-6,100,100],
        ['Pixels in each type','RFI-labeled within each type',"Each type’s share of validation"],
        ['Pixel locations (millions)',"% of that type’s pixels",'% of all validation pixels'],
        ['#39849b','#c8802b','#40856d']):
    vals = [r[field]*factor for r in breakdown]
    ax.barh(np.arange(len(breakdown)), vals, color=color)
    for i, value in enumerate(vals):
        label = f'{value:.1f}M' if field == 'pixel_locations' else f'{value:.3f}%' if field == 'rfi_fraction' else f'{value:.1f}%'
        ax.text(value + max(vals)*.015, i, label, va='center', fontsize=9)
    ax.set(title=title, xlabel=xlabel, xlim=(0,max(vals)*1.25))
    ax.grid(axis='x', alpha=.2)
axes[0].set_yticks(np.arange(len(breakdown)), [r['rfi_type'] for r in breakdown])
axes[0].invert_yaxis()
fig.suptitle(f'{total_pixels:,} pixel locations · validation only · '
             f"{sum(r['recordings'] for r in breakdown)} recordings · controls included")
fig.tight_layout()
save(fig, 'dataset_breakdown')


In [ ]:
header = '| Family | Recordings | Pixel locations | RFI pixels | RFI within family | Share of validation |\n|---|---:|---:|---:|---:|---:|'
rows = [f"| {r['rfi_type']} | {r['recordings']} | {r['pixel_locations']:,} | {r['rfi_pixels']:,} | {r['rfi_fraction']:.3%} | {r['dataset_fraction']:.1%} |" for r in breakdown]
display(Markdown(header + '\n' + '\n'.join(rows)))
print(f"Total: {total_pixels:,} pixels; RFI: {summary['positive_cells']:,} ({summary['positive_cells']/total_pixels:.3%})")

## 4. Error proportions: compare families fairly

The two panels use explicitly different denominators:

- **Miss rate = FN/(TP+FN):** missed pixels as a percentage of actual RFI pixels.
- **False-positive rate = FP/(FP+TN):** false alarms as a percentage of actual non-RFI pixels.

Each is computed within a family, avoiding the dependence of absolute counts on family size. Miss rate is undefined for clean and sky, so they are omitted from that panel. Axes use percentages and different ranges to keep FPR differences visible. These are pooled pixel rates, not equally weighted averages of recording rates.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, metric, title, denominator in zip(
        axes, ['miss_rate', 'fpr'],
        ['Missed RFI proportion by family', 'False-positive proportion by family'],
        ['FN / (TP + FN): % of actual RFI pixels',
         'FP / (FP + TN): % of actual non-RFI pixels']):
    order = sorted([f for f in all_families if np.isfinite(model[f][metric])],
                   key=lambda f: model[f][metric], reverse=True)
    vals = [100 * model[f][metric] for f in order]
    ax.barh(order, vals, color='#39849b')
    for i, value in enumerate(vals):
        ax.text(value + .01 * max(vals), i, f'{value:.2f}%', va='center', fontsize=9)
    ax.invert_yaxis()
    ax.set(title=title, xlabel=denominator, xlim=(0, max(vals)*1.22))
    ax.grid(axis='x', alpha=.2)
fig.suptitle('Validation error rates — clean/sky have no defined RFI miss rate')
fig.tight_layout()
save(fig, 'error_proportions')



## Interpretation
FSK and BPSK have high missed-RFI rates. Clean and sky controls have no positive labels, so their recall is undefined; report FPR instead. These descriptive differences do not establish a cause. Spectrogram error examples require the original data and checkpoint; use the grid workflow’s saved example plots when those inputs are available.


In [ ]:
print("Saved:", OUTPUT_DIR)
